<a href="https://colab.research.google.com/github/armandochernandez-ai/Curso-python-slava/blob/main/CUCEA/AZUCAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install selenium
import pandas as pd
import time
import re
import os
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import logging

# Configuración para Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✅ Ejecutando en Google Colab")
except:
    IN_COLAB = False
    print("❌ No se detectó Google Colab")

# Configuración de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class SNIIMExtractorAzucarCompleto:
    def __init__(self):
        self.base_url = "https://www.economia-sniim.gob.mx/Nuevo/Home.aspx?opcion=Consultas/MercadosNacionales/PreciosDeMercado/Agricolas/ConsultaAzucar.aspx?SubOpcion=7|0"
        self.driver = None
        self.wait = None

    def setup_driver(self):
        """Configura el WebDriver de Selenium"""
        try:
            if IN_COLAB:
                print("🔧 Configurando ChromeDriver para Colab...")
                !apt-get update > /dev/null 2>&1
                !apt-get install -y chromium-chromedriver > /dev/null 2>&1

                chrome_options = Options()
                chrome_options.add_argument('--headless')
                chrome_options.add_argument('--no-sandbox')
                chrome_options.add_argument('--disable-dev-shm-usage')
                chrome_options.add_argument('--disable-gpu')
                chrome_options.add_argument('--window-size=1920,1080')
                chrome_options.add_argument('--user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

                self.driver = webdriver.Chrome(options=chrome_options)
            else:
                chrome_options = Options()
                chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
                self.driver = webdriver.Chrome(options=chrome_options)

            self.wait = WebDriverWait(self.driver, 30)
            print("✅ ChromeDriver configurado correctamente")
            return True

        except Exception as e:
            logger.error(f"❌ Error configurando ChromeDriver: {e}")
            return False

    def navigate_to_form(self):
        """Navega al formulario principal para azúcar"""
        try:
            print("🔄 Navegando al formulario de azúcar...")

            # Cargar la página principal
            self.driver.get(self.base_url)
            time.sleep(8)

            # Cambiar al iframe que contiene el formulario
            iframes = self.driver.find_elements(By.TAG_NAME, "iframe")
            if not iframes:
                print("❌ No se encontraron iframes")
                return False

            self.driver.switch_to.frame(iframes[0])
            time.sleep(3)

            # Verificar que estamos en el formulario correcto para azúcar
            test_elements = ["ddlProducto", "ddlOrigen", "ddlDestino"]
            elementos_encontrados = 0
            for element_id in test_elements:
                try:
                    element = self.driver.find_element(By.ID, element_id)
                    if element:
                        elementos_encontrados += 1
                        print(f"✅ Elemento encontrado: {element_id}")
                except:
                    print(f"❌ No se pudo encontrar {element_id}")

            if elementos_encontrados >= 2:
                print("✅ Formulario de azúcar cargado correctamente")
                return True
            else:
                print("❌ Formulario de azúcar no se cargó correctamente")
                return False

        except Exception as e:
            logger.error(f"❌ Error navegando al formulario: {e}")
            return False

    def get_initial_page(self):
        return self.navigate_to_form()

    def get_productos(self):
        """Extrae la lista de productos del dropdown para azúcar"""
        try:
            print("📋 Extrayendo lista de productos de azúcar...")

            # Buscar dropdown de productos
            product_selectors = [
                "ctl00_ContentPlaceHolder1_ddlProducto",
                "ddlProducto"
            ]

            dropdown_element = None
            for selector in product_selectors:
                try:
                    dropdown_element = self.driver.find_element(By.ID, selector)
                    if dropdown_element:
                        print(f"✅ Dropdown de productos encontrado: {selector}")
                        break
                except:
                    continue

            if not dropdown_element:
                print("❌ No se pudo encontrar el dropdown de productos")
                return []

            # Crear objeto Select y obtener opciones
            select_productos = Select(dropdown_element)
            options = select_productos.options

            print(f"📊 Se encontraron {len(options)} opciones en el dropdown")

            productos = []
            for i, option in enumerate(options):
                value = option.get_attribute("value")
                text = option.text.strip()

                # Solo incluir opciones con valor y que no sea "Todos" o "Seleccione"
                if value and value != "" and value != "-1" and text and text != "Seleccione" and text != "Todos":
                    productos.append({
                        'id': value,
                        'nombre': text,
                        'index': i
                    })

                    # Mostrar los primeros 5 productos para verificación
                    if i < 5:
                        print(f"   {i+1:2d}. {text} (valor: {value})")

            if len(options) > 5:
                print(f"   ... y {len(options) - 5} productos más")

            print(f"✅ Se encontraron {len(productos)} productos válidos")
            return productos

        except Exception as e:
            logger.error(f"❌ Error extrayendo productos: {e}")
            return []

    def set_fechas(self, fecha_inicio, fecha_fin):
        """Establece las fechas de consulta"""
        try:
            print("📅 Configurando fechas de consulta...")

            # Para azúcar, hay dos campos de fecha
            fecha_inicio_selectors = ["txtFechaInicio", "ctl00_ContentPlaceHolder1_txtFechaInicio"]
            fecha_fin_selectors = ["txtFechaFinal", "ctl00_ContentPlaceHolder1_txtFechaFinal"]

            # Fecha inicio
            fecha_inicio_input = None
            for selector in fecha_inicio_selectors:
                try:
                    fecha_inicio_input = self.driver.find_element(By.ID, selector)
                    if fecha_inicio_input:
                        break
                except:
                    continue

            if fecha_inicio_input:
                fecha_inicio_input.clear()
                fecha_inicio_input.send_keys(fecha_inicio.strftime('%d/%m/%Y'))
                print(f"📅 Fecha inicial establecida: {fecha_inicio.strftime('%d/%m/%Y')}")
            else:
                print("❌ No se pudo encontrar campo fecha inicio")
                return False

            # Fecha fin
            fecha_fin_input = None
            for selector in fecha_fin_selectors:
                try:
                    fecha_fin_input = self.driver.find_element(By.ID, selector)
                    if fecha_fin_input:
                        break
                except:
                    continue

            if fecha_fin_input:
                fecha_fin_input.clear()
                fecha_fin_input.send_keys(fecha_fin.strftime('%d/%m/%Y'))
                print(f"📅 Fecha final establecida: {fecha_fin.strftime('%d/%m/%Y')}")
                return True
            else:
                print("❌ No se pudo encontrar campo fecha fin")
                return False

        except Exception as e:
            logger.error(f"❌ Error estableciendo fechas: {e}")
            return False

    def set_todos_ingenios_mercados(self):
        """Selecciona 'Todos' en Ingenio y Mercado de destino"""
        try:
            print("📍 Configurando Ingenios y Mercados...")

            # Buscar dropdown de Ingenio
            ingenio_selectors = [
                "ctl00_ContentPlaceHolder1_ddlOrigen",
                "ddlOrigen"
            ]

            for selector in ingenio_selectors:
                try:
                    ingenio_element = self.driver.find_element(By.ID, selector)
                    select_ingenio = Select(ingenio_element)

                    # Intentar seleccionar "Todos"
                    try:
                        select_ingenio.select_by_value("-1")
                        print(f"📍 Ingenio configurado: Todos")
                        break
                    except:
                        try:
                            select_ingenio.select_by_visible_text("Todos")
                            print("📍 Ingenio configurado: Todos")
                            break
                        except:
                            continue
                except:
                    continue

            # Buscar dropdown de Mercado de destino
            mercado_selectors = [
                "ctl00_ContentPlaceHolder1_ddlDestino",
                "ddlDestino"
            ]

            for selector in mercado_selectors:
                try:
                    mercado_element = self.driver.find_element(By.ID, selector)
                    select_mercado = Select(mercado_element)

                    # Intentar seleccionar "Todos"
                    try:
                        select_mercado.select_by_value("-1")
                        print(f"📍 Mercado configurado: Todos")
                        break
                    except:
                        try:
                            select_mercado.select_by_visible_text("Todos")
                            print("📍 Mercado configurado: Todos")
                            break
                        except:
                            continue
                except:
                    continue

            return True

        except Exception as e:
            logger.error(f"❌ Error configurando ingenios/mercados: {e}")
            return False

    def seleccionar_producto(self, producto_id):
        """Selecciona un producto específico"""
        try:
            product_selectors = [
                "ctl00_ContentPlaceHolder1_ddlProducto",
                "ddlProducto"
            ]

            dropdown_element = None
            for selector in product_selectors:
                try:
                    dropdown_element = self.driver.find_element(By.ID, selector)
                    if dropdown_element:
                        break
                except:
                    continue

            if not dropdown_element:
                return False

            select_productos = Select(dropdown_element)
            select_productos.select_by_value(producto_id)

            # Esperar a que la página procese la selección
            time.sleep(3)
            return True

        except Exception as e:
            logger.error(f"❌ Error seleccionando producto {producto_id}: {e}")
            return False

    def hacer_consulta(self):
        """Ejecuta la consulta y espera los resultados"""
        try:
            # Buscar botón de búsqueda
            boton_selectors = [
                "ctl00_ContentPlaceHolder1_btnBuscar",
                "btnBuscar",
                "//input[@type='submit' and contains(@value, 'Buscar')]"
            ]

            boton_buscar = None
            for selector in boton_selectors:
                try:
                    if selector.startswith("//"):
                        elements = self.driver.find_elements(By.XPATH, selector)
                        if elements:
                            boton_buscar = elements[0]
                            print(f"✅ Botón 'Buscar' encontrado con XPath: {selector}")
                            break
                    else:
                        boton_buscar = self.driver.find_element(By.ID, selector)
                        print(f"✅ Botón 'Buscar' encontrado con ID: {selector}")
                        break
                except:
                    continue

            if not boton_buscar:
                print("❌ No se pudo encontrar el botón 'Buscar'")
                return False

            # Desplazarse al botón si es necesario
            self.driver.execute_script("arguments[0].scrollIntoView(true);", boton_buscar)
            time.sleep(1)

            boton_buscar.click()
            print("🔄 Ejecutando búsqueda...")

            # Esperar a que los resultados se carguen
            time.sleep(10)

            # Verificar si hay resultados
            page_source = self.driver.page_source
            if "No se encontraron registros" in page_source:
                print("ℹ️ No se encontraron registros para esta consulta")
                return True
            elif "gvResultados" in page_source:
                print("✅ Resultados cargados correctamente")
                return True
            else:
                print("⚠️ No se pudo determinar el estado de la consulta")
                return True

        except Exception as e:
            logger.error(f"❌ Error ejecutando consulta: {e}")
            return False

    def hay_paginacion(self):
        """Verifica si hay paginación en los resultados"""
        try:
            # Buscar controles de paginación
            paginacion_selectors = [
                "//a[contains(@href, 'Page$')]",
                "//a[contains(text(), '...')]",
                "//table[contains(@id, 'gvResultados')]//tr[last()]//a"
            ]

            for selector in paginacion_selectors:
                try:
                    elementos = self.driver.find_elements(By.XPATH, selector)
                    if elementos:
                        print(f"✅ Se encontró paginación: {len(elementos)} elementos")
                        return True
                except:
                    continue

            print("ℹ️ No se encontró paginación")
            return False

        except Exception as e:
            print(f"❌ Error verificando paginación: {e}")
            return False

    def obtener_total_paginas(self):
        """Obtiene el número total de páginas"""
        try:
            # Buscar el texto que indica el total de páginas
            paginacion_text_selectors = [
                "//span[contains(text(), 'Página')]",
                "//td[contains(text(), 'Página')]",
                "//*[contains(text(), 'Página') and contains(text(), 'de')]"
            ]

            for selector in paginacion_text_selectors:
                try:
                    elemento = self.driver.find_element(By.XPATH, selector)
                    texto = elemento.text
                    # Extraer el número total de páginas del texto "Página X de Y"
                    match = re.search(r'Página\s*\d+\s*de\s*(\d+)', texto)
                    if match:
                        total_paginas = int(match.group(1))
                        print(f"📄 Total de páginas encontrado: {total_paginas}")
                        return total_paginas
                except:
                    continue

            # Si no encuentra el texto, contar los enlaces de página
            try:
                paginas = self.driver.find_elements(By.XPATH, "//a[contains(@href, 'Page$')]")
                if paginas:
                    # El último enlace suele ser el número más alto
                    numeros_paginas = []
                    for pagina in paginas:
                        try:
                            texto = pagina.text
                            if texto.isdigit():
                                numeros_paginas.append(int(texto))
                        except:
                            continue

                    if numeros_paginas:
                        total_paginas = max(numeros_paginas)
                        print(f"📄 Total de páginas estimado: {total_paginas}")
                        return total_paginas
            except:
                pass

            print("ℹ️ No se pudo determinar el total de páginas, asumiendo 1 página")
            return 1

        except Exception as e:
            print(f"❌ Error obteniendo total de páginas: {e}")
            return 1

    def ir_a_pagina(self, numero_pagina):
        """Navega a una página específica de resultados"""
        try:
            if numero_pagina == 1:
                return True  # Ya estamos en la primera página

            print(f"📄 Intentando ir a página {numero_pagina}...")

            # Buscar enlace de página específica
            selectors = [
                f"//a[contains(@href, 'Page${numero_pagina}')]",
                f"//a[text()='{numero_pagina}']",
                f"//a[contains(text(), '{numero_pagina}')]"
            ]

            for selector in selectors:
                try:
                    enlace = self.driver.find_element(By.XPATH, selector)
                    self.driver.execute_script("arguments[0].scrollIntoView(true);", enlace)
                    time.sleep(1)
                    enlace.click()
                    print(f"✅ Navegado a página {numero_pagina}")
                    time.sleep(5)  # Esperar a que cargue la nueva página
                    return True
                except:
                    continue

            # Intentar con el enlace "..." si existe
            if numero_pagina > 5:  # Asumiendo que solo muestra 5 páginas a la vez
                try:
                    enlace_puntos = self.driver.find_element(By.XPATH, "//a[contains(text(), '...')]")
                    self.driver.execute_script("arguments[0].scrollIntoView(true);", enlace_puntos)
                    time.sleep(1)
                    enlace_puntos.click()
                    time.sleep(5)
                    # Después de hacer clic en "...", intentar encontrar la página específica
                    return self.ir_a_pagina(numero_pagina)
                except:
                    pass

            print(f"❌ No se pudo navegar a la página {numero_pagina}")
            return False

        except Exception as e:
            print(f"❌ Error navegando a página {numero_pagina}: {e}")
            return False

    def extraer_datos_tabla_completa(self, producto_id, producto_nombre):
        """Extrae datos de TODAS las páginas de resultados"""
        try:
            datos_totales = []
            paginas_procesadas = 0

            # Primero verificar si hay paginación
            if not self.hay_paginacion():
                print("ℹ️ Solo hay una página de resultados")
                datos_pagina = self.extraer_datos_pagina_actual(producto_id, producto_nombre)
                datos_totales.extend(datos_pagina)
                paginas_procesadas = 1
            else:
                # Obtener el total de páginas
                total_paginas = self.obtener_total_paginas()
                print(f"📊 Procesando {total_paginas} páginas de resultados...")

                # Procesar cada página
                for pagina in range(1, total_paginas + 1):
                    print(f"\n📄 Procesando página {pagina}/{total_paginas}...")

                    # Navegar a la página (para la primera página ya estamos ahí)
                    if pagina > 1:
                        if not self.ir_a_pagina(pagina):
                            print(f"❌ No se pudo acceder a la página {pagina}, continuando...")
                            continue

                    # Extraer datos de la página actual
                    datos_pagina = self.extraer_datos_pagina_actual(producto_id, producto_nombre)
                    datos_totales.extend(datos_pagina)
                    paginas_procesadas += 1

                    print(f"✅ Página {pagina}: {len(datos_pagina)} registros")

                    # Pequeña pausa entre páginas
                    if pagina < total_paginas:
                        time.sleep(2)

            print(f"📊 Total: {len(datos_totales)} registros de {paginas_procesadas} páginas")
            return datos_totales

        except Exception as e:
            logger.error(f"❌ Error extrayendo datos completos: {e}")
            return []

    def extraer_datos_pagina_actual(self, producto_id, producto_nombre):
        """Extrae los datos de la página actual de resultados"""
        try:
            datos = []

            # Verificar si hay mensaje de "no hay registros"
            if "No se encontraron registros" in self.driver.page_source:
                print("   ℹ️ No se encontraron registros en esta página")
                return datos

            # Buscar tabla de resultados
            tabla_selectors = [
                "ctl00_ContentPlaceHolder1_gvResultados",
                "gvResultados",
                "//table[contains(@id, 'Resultados')]"
            ]

            tabla_element = None
            for selector in tabla_selectors:
                try:
                    if selector.startswith("//"):
                        tabla_element = self.driver.find_element(By.XPATH, selector)
                    else:
                        tabla_element = self.driver.find_element(By.ID, selector)
                    if tabla_element:
                        break
                except:
                    continue

            if not tabla_element:
                print("   ℹ️ No se encontró tabla de resultados en esta página")
                return datos

            # Obtener todas las filas de la tabla
            filas = tabla_element.find_elements(By.TAG_NAME, "tr")
            print(f"   📊 Encontradas {len(filas)} filas en la tabla")

            # Variables para rastrear el mercado actual
            mercado_actual = ""
            producto_actual = producto_nombre

            # Procesar cada fila
            for i, fila in enumerate(filas):
                try:
                    celdas = fila.find_elements(By.TAG_NAME, "td")

                    if len(celdas) == 1:
                        # Podría ser un encabezado de sección (mercado)
                        texto_celda = celdas[0].text.strip()
                        if ":" in texto_celda and not re.match(r'\d{1,2}/\d{1,2}/\d{4}', texto_celda):
                            mercado_actual = texto_celda
                            print(f"   🏪 Mercado encontrado: {mercado_actual}")
                            continue

                    elif len(celdas) >= 3:
                        # Podría ser una fila de datos
                        fecha = celdas[0].text.strip()

                        # Validar formato de fecha
                        if not re.match(r'\d{1,2}/\d{1,2}/\d{4}', fecha):
                            # Podría ser el nombre del producto en esta fila
                            if celdas[0].text.strip() and not "Página" in celdas[0].text:
                                producto_actual = celdas[0].text.strip()
                            continue

                        # Es una fila de datos válida
                        ingenio = celdas[1].text.strip() if len(celdas) > 1 else ''
                        precio = celdas[2].text.strip() if len(celdas) > 2 else ''
                        observaciones = celdas[3].text.strip() if len(celdas) > 3 else ''

                        dato = {
                            'producto_id': producto_id,
                            'producto_nombre': producto_actual,
                            'mercado': mercado_actual,
                            'fecha': fecha,
                            'ingenio': ingenio,
                            'precio_encuestado': self.limpiar_precio(precio),
                            'observaciones': observaciones,
                            'fecha_consulta': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        }

                        # Solo agregar si tiene datos válidos
                        if dato['fecha'] and dato['ingenio'] and dato['precio_encuestado']:
                            datos.append(dato)

                except Exception as e:
                    continue

            print(f"   ✅ {len(datos)} registros extraídos de esta página")
            return datos

        except Exception as e:
            print(f"   ❌ Error extrayendo datos de página: {e}")
            return []

    def limpiar_precio(self, precio_str):
        """Limpia y convierte el precio a float"""
        if not precio_str:
            return None

        # Remover caracteres no numéricos excepto punto decimal
        precio_limpio = re.sub(r'[^\d.]', '', precio_str)

        try:
            return float(precio_limpio) if precio_limpio else None
        except ValueError:
            return None

    def guardar_datos(self, datos, directorio_salida, archivo_nombre):
        """Guarda los datos en archivos CSV"""
        if not datos:
            logger.warning("⚠️ No hay datos para guardar")
            return None

        os.makedirs(directorio_salida, exist_ok=True)

        # Crear DataFrame
        df = pd.DataFrame(datos)

        # Nombre del archivo
        archivo_csv = os.path.join(directorio_salida, f"{archivo_nombre}.csv")

        # Guardar en CSV
        df.to_csv(archivo_csv, index=False, encoding='utf-8-sig')
        logger.info(f"💾 Datos guardados en: {archivo_csv}")

        return archivo_csv

    def reset_form(self):
        """Vuelve al formulario principal después de una consulta"""
        try:
            print("🔄 Restableciendo formulario...")

            # Volver al contexto principal
            self.driver.switch_to.default_content()

            # Navegar de nuevo al formulario
            return self.navigate_to_form()

        except Exception as e:
            logger.error(f"❌ Error restableciendo formulario: {e}")
            return False

    def close(self):
        """Cierra el navegador"""
        if self.driver:
            try:
                self.driver.switch_to.default_content()
            except:
                pass
            self.driver.quit()
            print("🔒 Navegador cerrado")

def configurar_directorio_colab():
    """Configura Google Drive para Colab"""
    if IN_COLAB:
        print("📁 Montando Google Drive...")
        drive.mount('/content/drive')

        # Crear directorio base
        directorio_base = "/content/drive/MyDrive/AZUCAR"
        os.makedirs(directorio_base, exist_ok=True)

        print(f"✅ Directorio configurado: {directorio_base}")
        return directorio_base
    else:
        # Directorio local para pruebas
        directorio_local = "AZUCAR"
        os.makedirs(directorio_local, exist_ok=True)
        print(f"📂 Directorio local: {directorio_local}")
        return directorio_local

def main_azucar_completo():
    """Función principal para azúcar con paginación completa"""
    print("=" * 70)
    print("🍭 EXTRACTOR DE DATOS SNIIM - AZÚCAR COMPLETO")
    print("📍 Con manejo de paginación y estructura real de datos")
    print("=" * 70)

    # Configurar directorio de salida
    directorio_salida = configurar_directorio_colab()

    # Inicializar extractor
    extractor = SNIIMExtractorAzucarCompleto()

    # Configurar el driver
    if not extractor.setup_driver():
        print("❌ No se pudo configurar Selenium. Terminando ejecución.")
        return

    try:
        # Obtener página inicial
        print("\n🔗 Navegando a la página del SNIIM Azúcar...")
        if not extractor.get_initial_page():
            print("❌ No se pudo cargar la página inicial.")
            return

        # Obtener lista de productos
        print("\n📋 Extrayendo lista de productos de azúcar...")
        productos = extractor.get_productos()

        if not productos:
            print("❌ No se encontraron productos disponibles.")
            return

        # Configurar fechas (últimos 7 días para prueba)
        fecha_fin = datetime.now() - timedelta(days=5)
        fecha_inicio = fecha_fin - timedelta(days=2)

        print(f"\n📅 Rango de fechas: {fecha_inicio.strftime('%d/%m/%Y')} - {fecha_fin.strftime('%d/%m/%Y')}")
        print(f"🍭 Productos disponibles: {len(productos)}")

        # Procesar productos (limitar para prueba inicial)
        productos_a_procesar = productos[:3]  # Solo 3 productos para prueba
        print(f"🔍 Procesando {len(productos_a_procesar)} productos...")

        # Recolectar todos los datos
        print(f"\n🎬 INICIANDO EXTRACCIÓN COMPLETA DE AZÚCAR...")

        todos_los_datos = []
        productos_exitosos = 0

        for i, producto in enumerate(productos_a_procesar, 1):
            print(f"\n{'='*60}")
            print(f"🍭 [{i}/{len(productos_a_procesar)}] PROCESANDO: {producto['nombre']}")
            print(f"{'='*60}")

            try:
                # Configurar fechas para cada consulta
                if not extractor.set_fechas(fecha_inicio, fecha_fin):
                    print("❌ Error configurando fechas, continuando...")
                    continue

                # Configurar ingenios y mercados como "Todos"
                if not extractor.set_todos_ingenios_mercados():
                    print("❌ Error configurando ingenios/mercados, continuando...")
                    continue

                # Seleccionar producto
                if not extractor.seleccionar_producto(producto['id']):
                    print("❌ Error seleccionando producto, continuando...")
                    continue

                # Ejecutar consulta
                if not extractor.hacer_consulta():
                    print("❌ Error en consulta, continuando...")
                    continue

                # Extraer datos de TODAS las páginas
                datos_producto = extractor.extraer_datos_tabla_completa(
                    producto['id'],
                    producto['nombre']
                )

                if datos_producto:
                    todos_los_datos.extend(datos_producto)
                    productos_exitosos += 1
                    print(f"✅ {len(datos_producto)} registros obtenidos en total")
                else:
                    print("ℹ️ Sin registros para este producto")

                # Restablecer formulario después de cada consulta
                if not extractor.reset_form():
                    print("❌ No se pudo restablecer el formulario, intentando continuar...")
                    extractor.close()
                    time.sleep(5)
                    if not extractor.setup_driver() or not extractor.get_initial_page():
                        print("❌ No se pudo recuperar la sesión, terminando ejecución.")
                        break

            except Exception as e:
                print(f"❌ Error inesperado procesando {producto['nombre']}: {e}")
                # Intentar recuperar el formulario
                if not extractor.reset_form():
                    extractor.close()
                    time.sleep(5)
                    if not extractor.setup_driver() or not extractor.get_initial_page():
                        print("❌ No se pudo recuperar la sesión, terminando ejecución.")
                        break

            # Esperar entre consultas
            time.sleep(3)

        # Guardar datos finales
        if todos_los_datos:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            archivo_final = extractor.guardar_datos(
                todos_los_datos,
                directorio_salida,
                f"azucar_completo_{timestamp}"
            )

            print(f"\n{'='*80}")
            print("🎉 EXTRACCIÓN COMPLETA DE AZÚCAR FINALIZADA")
            print(f"{'='*80}")
            print(f"📊 TOTAL DE REGISTROS: {len(todos_los_datos):,}")
            print(f"🍭 PRODUCTOS EXITOSOS: {productos_exitosos}/{len(productos_a_procesar)}")
            print(f"📅 PERÍODO: {fecha_inicio.strftime('%d/%m/%Y')} - {fecha_fin.strftime('%d/%m/%Y')}")
            print(f"💾 ARCHIVO GUARDADO: {archivo_final}")

            # Mostrar resumen de datos
            df = pd.DataFrame(todos_los_datos)
            print(f"\n📋 RESUMEN DE DATOS:")
            print(f"   • Mercados únicos: {df['mercado'].nunique()}")
            print(f"   • Ingenios únicos: {df['ingenio'].nunique()}")
            print(f"   • Rango de fechas: {df['fecha'].min()} a {df['fecha'].max()}")
            print(f"   • Rango de precios: {df['precio_encuestado'].min():.2f} - {df['precio_encuestado'].max():.2f}")

        else:
            print("\n❌ No se extrajeron datos.")

    except Exception as e:
        logger.error(f"❌ Error en la ejecución principal: {str(e)}")
        import traceback
        traceback.print_exc()

        # Intentar guardar datos recolectados hasta el momento
        if 'todos_los_datos' in locals() and todos_los_datos:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            archivo_error = extractor.guardar_datos(
                todos_los_datos,
                directorio_salida,
                f"azucar_parcial_{timestamp}"
            )
            print(f"💾 Datos guardados hasta el error: {archivo_error}")

    finally:
        extractor.close()

if __name__ == "__main__":
    # Instalar Selenium si es necesario en Colab
    if IN_COLAB:
        print("📦 Instalando Selenium...")
        !pip install selenium -q

    main_azucar_completo()

✅ Ejecutando en Google Colab
📦 Instalando Selenium...
🍭 EXTRACTOR DE DATOS SNIIM - AZÚCAR COMPLETO
📍 Con manejo de paginación y estructura real de datos
📁 Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Directorio configurado: /content/drive/MyDrive/AZUCAR
🔧 Configurando ChromeDriver para Colab...
✅ ChromeDriver configurado correctamente

🔗 Navegando a la página del SNIIM Azúcar...
🔄 Navegando al formulario de azúcar...
✅ Elemento encontrado: ddlProducto
✅ Elemento encontrado: ddlOrigen
✅ Elemento encontrado: ddlDestino
✅ Formulario de azúcar cargado correctamente

📋 Extrayendo lista de productos de azúcar...
📋 Extrayendo lista de productos de azúcar...
✅ Dropdown de productos encontrado: ddlProducto
📊 Se encontraron 3 opciones en el dropdown
    2. Azúcar Estándar - (valor: 156)
    3. Azúcar Refinada - (valor: 155)
✅ Se encontraron 2 productos válidos

📅 Rango de fechas: 29/10/